## Exercícios com Pandas e Polars

In [1]:
data = [
    {'nome': 'Ana', 'idade': 21, 'estado': 'SP', 'salario': 7600},
    {'nome': 'Francieli', 'idade': 29, 'estado': 'SP', 'salario': 8500},
    {'nome': 'Andre', 'idade': 31, 'estado': 'RJ', 'salario': 9200},
    {'nome': 'Amanda', 'idade': 32, 'estado': 'MG', 'salario': 6900}
]

In [2]:
print(data)

[{'nome': 'Ana', 'idade': 21, 'estado': 'SP', 'salario': 7600}, {'nome': 'Francieli', 'idade': 29, 'estado': 'SP', 'salario': 8500}, {'nome': 'Andre', 'idade': 31, 'estado': 'RJ', 'salario': 9200}, {'nome': 'Amanda', 'idade': 32, 'estado': 'MG', 'salario': 6900}]


## Exercício 1 - Filtrar dados

In [4]:
import pandas as pd

df_pd = pd.DataFrame(data)

df_filtered = df_pd[df_pd['idade'] > 30]

print(df_filtered)

     nome  idade estado  salario
2   Andre     31     RJ     9200
3  Amanda     32     MG     6900


In [5]:
!pip install polars

In [6]:
import polars as pl

df_pl = pl.DataFrame(data)

df_filtered = df_pl.filter(pl.col('idade') > 30)

print(df_filtered)

shape: (2, 4)
┌────────┬───────┬────────┬─────────┐
│ nome   ┆ idade ┆ estado ┆ salario │
│ ---    ┆ ---   ┆ ---    ┆ ---     │
│ str    ┆ i64   ┆ str    ┆ i64     │
╞════════╪═══════╪════════╪═════════╡
│ Andre  ┆ 31    ┆ RJ     ┆ 9200    │
│ Amanda ┆ 32    ┆ MG     ┆ 6900    │
└────────┴───────┴────────┴─────────┘


## Exercício 2 - Criar nova coluna
**Criar coluna salario_anual = salario * 12**

In [9]:
df_pd["salario_anual"] = df_pd["salario"] * 12

print(df_pd)

        nome  idade estado  salario  salario_anual
0        Ana     21     SP     7600          91200
1  Francieli     29     SP     8500         102000
2      Andre     31     RJ     9200         110400
3     Amanda     32     MG     6900          82800


In [11]:
df_pl = df_pl.with_columns(
    (pl.col('salario') * 12).alias('salario_anual')
)

print(df_pl)

shape: (4, 5)
┌───────────┬───────┬────────┬─────────┬───────────────┐
│ nome      ┆ idade ┆ estado ┆ salario ┆ salario_anual │
│ ---       ┆ ---   ┆ ---    ┆ ---     ┆ ---           │
│ str       ┆ i64   ┆ str    ┆ i64     ┆ i64           │
╞═══════════╪═══════╪════════╪═════════╪═══════════════╡
│ Ana       ┆ 21    ┆ SP     ┆ 7600    ┆ 91200         │
│ Francieli ┆ 29    ┆ SP     ┆ 8500    ┆ 102000        │
│ Andre     ┆ 31    ┆ RJ     ┆ 9200    ┆ 110400        │
│ Amanda    ┆ 32    ┆ MG     ┆ 6900    ┆ 82800         │
└───────────┴───────┴────────┴─────────┴───────────────┘


## Exercício 3 — Agrupamento
**Média de salário por cidade**

In [13]:
df_pd_grouped = df_pd.groupby('estado')['salario'].mean()

print(df_pd_grouped)

estado
MG    6900.0
RJ    9200.0
SP    8050.0
Name: salario, dtype: float64


In [14]:
df_pl_grouped = df_pl.group_by('estado').agg(
    pl.col('salario').mean()
)

print(df_pl_grouped)

shape: (3, 2)
┌────────┬─────────┐
│ estado ┆ salario │
│ ---    ┆ ---     │
│ str    ┆ f64     │
╞════════╪═════════╡
│ RJ     ┆ 9200.0  │
│ SP     ┆ 8050.0  │
│ MG     ┆ 6900.0  │
└────────┴─────────┘


## Exercício 5 — Pipeline
- Filtrar idade > 25
- Criar salário anual
- Agrupar por cidade

In [16]:
df_pd_pipeline = (
    df_pd[df_pd['idade'] > 25]
    .assign(salario_anual=lambda x: x['salario'] * 12)
    .groupby('estado')['salario_anual'].mean()
)

print(df_pd_pipeline)

estado
MG     82800.0
RJ    110400.0
SP    102000.0
Name: salario_anual, dtype: float64


In [17]:
df_pl_pipeline = (
    df_pl
    .filter(pl.col('idade') > 25)
    .with_columns(
        (pl.col('salario') * 12).alias('salario_anual')
    )
    .group_by('estado')
    .agg(pl.col('salario_anual').mean())
)

print(df_pl_pipeline)

shape: (3, 2)
┌────────┬───────────────┐
│ estado ┆ salario_anual │
│ ---    ┆ ---           │
│ str    ┆ f64           │
╞════════╪═══════════════╡
│ RJ     ┆ 110400.0      │
│ MG     ┆ 82800.0       │
│ SP     ┆ 102000.0      │
└────────┴───────────────┘


## Extra - Lazy execution no Polars

In [19]:
df_lazy = df_pl.lazy()

result = (
    df_lazy
    .filter(pl.col('idade') > 25)
    .with_columns((pl.col('salario') * 12).alias('salario_anual'))
    .group_by('estado')
    .agg(pl.col('salario_anual').mean())
    .collect()
)

print(result)

shape: (3, 2)
┌────────┬───────────────┐
│ estado ┆ salario_anual │
│ ---    ┆ ---           │
│ str    ┆ f64           │
╞════════╪═══════════════╡
│ RJ     ┆ 110400.0      │
│ MG     ┆ 82800.0       │
│ SP     ┆ 102000.0      │
└────────┴───────────────┘
